In [ ]:
from pyspark.sql import SparkSession
from pyspark import SparkConf

conf = SparkConf()
conf.set('spark.app.name', 'PySpark Partitioning')
conf.set('spark.master', 'local[*]')
# Change Warehouse Path
# conf.set('spark.sql.warehouse.dir', new_path)

spark = SparkSession.builder\
        .config(conf = conf)\
        .enableHiveSupport()\
        .getOrCreate()

In [ ]:
# Check my Warehouse Path

spark.conf.get("spark.sql.warehouse.dir")

In [7]:
df = spark.read.csv('appl_stock.csv', header = True, inferSchema = True)

df.printSchema()
df.show(5)

root
 |-- Date: date (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: integer (nullable = true)
 |-- Adj Close: double (nullable = true)

+----------+----------+----------+------------------+------------------+---------+------------------+
|      Date|      Open|      High|               Low|             Close|   Volume|         Adj Close|
+----------+----------+----------+------------------+------------------+---------+------------------+
|2010-01-04|213.429998|214.499996|212.38000099999996|        214.009998|123432400|         27.727039|
|2010-01-05|214.599998|215.589994|        213.249994|        214.379993|150476200|27.774976000000002|
|2010-01-06|214.379993|    215.23|        210.750004|        210.969995|138040000|27.333178000000004|
|2010-01-07|    211.75|212.000006|        209.050005|            210.58|119282800|          27.28265|
|2010-01-08|210.299994

In [8]:
import pyspark.sql.functions as f

df = df.withColumn('year', f.year(f.col('Date')))\
        .withColumn('month', f.month(f.col('Date')))

In [10]:
spark.sql('DROP TABLE IF EXISTS appl_stock')

DataFrame[]

In [11]:
df.write.partitionBy('year', 'month')\
        .saveAsTable('appl_stock')

In [ ]:
df = spark.read.table("appl_stock").where("year = 2016 and month = 12")

df.show(10)